# 05 — Analysis, tables and figures  *(CPU only)*

Reads `03_nowrite_reproduction.json` and `04_retention_rows.json`. No GPU, no model.
Download those two files from the GPU box and run this locally.

**Produces:** Table 5 (RQ1), Table 6 (retention summary with R²), Table 7 (pairwise
half-life ratios), Table 8 (RQ3 correlations), Table 9 (variance decomposition), and
Figures 3–8.

Every figure written here is **real data**. The synthetic `ILLUSTRATIVE`-watermarked
figures in `Expected_Tables_and_Figures.docx` exist only to show intended shape and must
not be reused once these exist — that document says so explicitly.


In [ ]:
# --- bootstrap -------------------------------------------------------------------
# Upload `ahn_interp.py` next to this notebook (or anywhere up the tree).
import os, sys, json, importlib

def _find_module(name="ahn_interp.py", depth=4):
    here = os.getcwd()
    for _ in range(depth):
        for cand in (here, os.path.join(here, "src"), os.path.join(here, "notebooks")):
            if os.path.exists(os.path.join(cand, name)):
                return cand
        here = os.path.dirname(here)
    return None

_root = _find_module()
assert _root, ("ahn_interp.py not found. Upload it into this notebook's directory "
               "(Jupyter: Upload button, top right of the file browser).")
if _root not in sys.path:
    sys.path.insert(0, _root)

# Pin the working directory to wherever ahn_interp.py actually lives (normally the
# repo root). Without this, relative paths in CFG (results_dir, configs/*.json) resolve
# against whatever directory Jupyter happened to open in -- e.g. running this notebook
# from inside notebooks/ silently writes results to notebooks/results/... instead of
# results/... at the repo root, which is where every other notebook and 05's analysis
# step expect to find them.
os.chdir(_root)

import ahn_interp as ai
importlib.reload(ai)
ai.set_seed()
print("ahn_interp loaded from", _root)
print("working directory pinned to", os.getcwd())


In [ ]:
import json, os, numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

RUN = "results/run_3b_gdn"     # <- point at the folder you downloaded
ai.set_results_dir(RUN)
FIGDIR = os.path.join(RUN, "figures"); os.makedirs(FIGDIR, exist_ok=True)

data = ai.load_json("04_retention_rows.json")
rows = data["rows"]
main = [r for r in rows if not r["in_window"] and not r["shuffled"]]
print(f"{len(rows)} rows | {len(main)} main | readout: {rows[0]['readout']}")
print("needles:", sorted({r['needle'] for r in rows}))
print("layers:", sorted({r['layer'] for r in rows}))

try:
    repro = ai.load_json("03_nowrite_reproduction.json")
    print("\nNOWRITE reproduction present:", repro["summary"]["reproduction_ok"])
except FileNotFoundError:
    repro = None
    print("\n! no 03_nowrite_reproduction.json — RQ1 and RQ3 sections will be skipped")

CELL, SCALE = data["cfg"]["cell"], data["cfg"]["scale"]


## Figure 4 — the primary RQ2 figure

Log y-axis: exponential decay is a straight line here. **If it is not straight, the
half-life column of Table 6 is invalid** and has to be replaced by a non-parametric
statistic. Inspect this before filling Table 6 in.


In [ ]:
def agg(rs, key):
    """mean +/- bootstrap CI of `key` grouped by eviction distance"""
    by = {}
    for r in rs:
        by.setdefault(r["eviction_distance"], []).append(r[key])
    ds = sorted(by)
    pts = [ai.bootstrap_ci(by[d], n_boot=2000) for d in ds]
    return np.array(ds), np.array([p[0] for p in pts]), \
           np.array([p[1] for p in pts]), np.array([p[2] for p in pts])

layers = sorted({r["layer"] for r in main})
fig, ax = plt.subplots(figsize=(8, 5))
for L in layers:
    d, m, lo, hi = agg([r for r in main if r["layer"] == L], "p_mem")
    ax.plot(d, m, marker="o", label=f"layer {L}")
    ax.fill_between(d, lo, hi, alpha=0.18)
inwin = [r for r in rows if r["in_window"]]
if inwin:
    ax.axhline(np.mean([r["p_mem"] for r in inwin]), ls="--", c="k", alpha=.6,
               label="pre-eviction ceiling (C4)")
ax.set_yscale("log")
ax.set_xlabel("eviction distance (tokens past the compression boundary)")
ax.set_ylabel(r"$P_{\mathrm{mem}}(y^*)$")
ax.set_title(f"Figure 4 — retention decay · Qwen2.5-{SCALE} + AHN-{CELL} · {rows[0]['readout']}")
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{FIGDIR}/fig4_retention_decay.png", dpi=150)
print("saved fig4"); plt.close(fig)


## Figure 5 — rank, which is what a reader can actually parse

"the needle leaves the top-1000 after N tokens" is a sentence anyone understands;
"P_mem fell to 3e-5" is not.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for L in layers:
    d, m, lo, hi = agg([r for r in main if r["layer"] == L], "rank")
    ax.plot(d, m, marker="o", label=f"layer {L}")
    ax.fill_between(d, lo, hi, alpha=0.18)
ax.axhline(1000, ls=":", c="crimson", label="top-1000")
if inwin:
    ax.axhline(np.mean([r["rank"] for r in inwin]), ls="--", c="k", alpha=.6,
               label="pre-eviction ceiling (C4)")
ax.set_yscale("log"); ax.invert_yaxis()
ax.set_xlabel("eviction distance (tokens)"); ax.set_ylabel("rank of $y^*$  (lower = retained)")
ax.set_title(f"Figure 5 — target rank vs eviction distance · AHN-{CELL} {SCALE}")
ax.legend(); ax.grid(alpha=.3)
fig.tight_layout(); fig.savefig(f"{FIGDIR}/fig5_rank_decay.png", dpi=150)
print("saved fig5"); plt.close(fig)


## Figure 6 — layer profile, and Figure 7 — readout entropy

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Fig 6: retention at a fixed eviction distance, across depth
target_d = sorted({r["eviction_distance"] for r in main})[len(layers)//2 or 1]
close = lambda r: abs(r["eviction_distance"] - target_d) <= max(256, target_d * .25)
prof = {}
for r in main:
    if close(r):
        prof.setdefault(r["layer"], []).append(r["rank"])
ls = sorted(prof)
axes[0].bar(ls, [np.mean(prof[l]) for l in ls], color="crimson", alpha=.75)
axes[0].set_xlabel("layer"); axes[0].set_ylabel(f"mean rank @ ~{target_d} tok")
axes[0].invert_yaxis()
axes[0].set_title("Figure 6 — retention by depth")
axes[0].grid(alpha=.3, axis="y")

# Fig 7: entropy — blank memory (entropy rises) vs confidently wrong (entropy stays low)
for L in layers:
    d, m, lo, hi = agg([r for r in main if r["layer"] == L], "entropy")
    axes[1].plot(d, m, marker="o", label=f"layer {L}")
    axes[1].fill_between(d, lo, hi, alpha=.18)
axes[1].set_xlabel("eviction distance (tokens)"); axes[1].set_ylabel("readout entropy (nats)")
axes[1].set_title("Figure 7 — sharp memory vs diffuse memory")
axes[1].legend(); axes[1].grid(alpha=.3)

fig.tight_layout(); fig.savefig(f"{FIGDIR}/fig6_7_layer_and_entropy.png", dpi=150)
print("saved fig6+7"); plt.close(fig)


## Table 6 — retention summary, with the R² that decides its validity

In [ ]:
table6 = []
for L in layers:
    rs = [r for r in main if r["layer"] == L]
    by = {}
    for r in rs:
        by.setdefault(r["eviction_distance"], []).append(r["p_mem"])
    d = sorted(by); v = [float(np.mean(by[x])) for x in d]
    fit = ai.fit_exponential_halflife(d, v)
    r8k  = [r["rank"] for r in rs if abs(r["eviction_distance"] - 8192) < 2048]
    r32k = [r["rank"] for r in rs if abs(r["eviction_distance"] - 32768) < 8192]
    table6.append({
        "scale": SCALE, "cell": CELL, "layer": L,
        "half_life_tok": fit["half_life"] if fit["fit_adequate"] else None,
        "half_life_ci": fit["ci"] if fit["fit_adequate"] else None,
        "nonparametric_half_distance": fit["nonparametric_half_distance"],
        "fit_r2": fit["r2"], "fit_adequate": fit["fit_adequate"],
        "rank_at_8k": float(np.mean(r8k)) if r8k else None,
        "rank_at_32k": float(np.mean(r32k)) if r32k else None,
        "n_points": fit["n"],
    })
    print(json.dumps(table6[-1], indent=2, default=str))

if not any(t["fit_adequate"] for t in table6):
    print("\n!! NO adequate exponential fit at any layer.")
    print("   Per Expected_Tables_and_Figures, 'half-life' is then a meaningless summary")
    print("   and Table 6 must be replaced by the non-parametric column. Use")
    print("   `nonparametric_half_distance` and say so in the caption.")
ai.save_json(table6, "05_table6_retention_summary.json")


## Table 7 — pairwise half-life comparisons

Fill this in once you have **more than one cell**. The proposal's headline claim is a
ratio ≥ 2 with a CI excluding 1 on at least one pair — that is a difference with its own
interval, not something you read off two rows of Table 6. Three comparisons means a
multiplicity correction; report Holm-adjusted intervals alongside the raw ones.


In [ ]:
CELL_RUNS = {
    # "GatedDeltaNet": "results/run_3b_gdn/04_retention_rows.json",
    # "DeltaNet":      "results/run_3b_dn/04_retention_rows.json",
    # "Mamba2":        "results/run_3b_m2/04_retention_rows.json",
}
COMPARE_LAYER = 18

if len(CELL_RUNS) < 2:
    print("Table 7 needs >= 2 cells. Run notebook 04 for DeltaNet and Mamba2, then")
    print("fill CELL_RUNS above and re-run this cell.")
else:
    import itertools
    hl = {}
    for cell, path in CELL_RUNS.items():
        rs = [r for r in json.load(open(path))["rows"]
              if not r["in_window"] and not r["shuffled"] and r["layer"] == COMPARE_LAYER]
        by = {}
        for r in rs:
            by.setdefault(r["eviction_distance"], []).append(r["p_mem"])
        boots = []
        rng = np.random.default_rng(ai.SEED)
        for _ in range(2000):
            d = sorted(by)
            v = [float(np.mean(rng.choice(by[x], size=len(by[x])))) for x in d]
            f = ai.fit_exponential_halflife(d, v, n_boot=1)
            if np.isfinite(f["half_life"]):
                boots.append(f["half_life"])
        hl[cell] = np.array(boots)

    table7 = []
    for a, b in itertools.combinations(sorted(hl), 2):
        n = min(len(hl[a]), len(hl[b]))
        ratio = hl[a][:n] / np.clip(hl[b][:n], 1e-9, None)
        lo, hi = np.quantile(ratio, [.025, .975])
        table7.append({
            "comparison": f"{a} / {b}", "layer": COMPARE_LAYER,
            "ratio": float(np.median(ratio)), "ci": [float(lo), float(hi)],
            "difference_tok": float(np.median(hl[a][:n] - hl[b][:n])),
            "excludes_1": bool(lo > 1 or hi < 1),
            "meets_2x_criterion": bool(np.median(ratio) >= 2 and (lo > 1 or hi < 1)),
        })
        print(json.dumps(table7[-1], indent=2))
    ai.save_json(table7, "05_table7_pairwise_halflife.json")
    print("\nNOTE: 3 comparisons -> apply Holm and report both adjusted and raw.")


## Table 5 + Figure 3 — RQ1 behavioural effect

In [ ]:
if repro is None:
    print("skipped — run notebook 03 first")
else:
    per = repro["per_example"]
    table5 = []
    for s in ("short", "mid", "long"):
        rs = [r for r in per if r["stratum"] == s]
        if not rs: continue
        table5.append({
            "scale": SCALE, "cell": CELL, "stratum": s, "n": len(rs),
            "mean_f1": ai.bootstrap_ci([r["f1_ahn"] for r in rs])[0],
            "delta_f1_vs_nowrite": ai.bootstrap_ci([r["delta_f1"] for r in rs]),
            "answer_change_rate": float(np.mean([r["answer_changed"] for r in rs])),
        })
        print(json.dumps(table5[-1], indent=2, default=str))
    ai.save_json(table5, "05_table5_rq1.json")

    fig, ax = plt.subplots(figsize=(7, 3.4))
    ys = np.arange(len(table5))
    pts = [t["delta_f1_vs_nowrite"] for t in table5]
    ax.errorbar([p[0] for p in pts], ys,
                xerr=[[p[0]-p[1] for p in pts], [p[2]-p[0] for p in pts]],
                fmt="o", capsize=4, color="crimson")
    ax.axvline(0, c="k", lw=1)
    ax.set_yticks(ys); ax.set_yticklabels([f"{t['stratum']} (n={t['n']})" for t in table5])
    ax.set_xlabel(r"$\Delta$F1 (AHN $-$ NOWRITE)")
    ax.set_title(f"Figure 3 — RQ1 effect sizes · AHN-{CELL} {SCALE}")
    ax.grid(alpha=.3, axis="x")
    fig.tight_layout(); fig.savefig(f"{FIGDIR}/fig3_rq1_forest.png", dpi=150)
    print("saved fig3"); plt.close(fig)


## Table 8 + Figure 8 — RQ3, the paper's central test

This needs retention and ΔF1 **measured on the same examples**. Right now they are not:
notebook 03 runs LongBench-E HotpotQA and notebook 04 runs synthetic NIAH prompts. The
join key does not exist yet.

**This is the single most important next piece of engineering.** See the "Next steps"
section of the README: notebook 04 has to be extended to run on the *same* LongBench-E
examples as notebook 03, reading out a needle-equivalent target token per example — for
HotpotQA, the first token of the gold answer.

The cell below is written and ready; it activates the moment a joined file exists.


In [ ]:
JOINED = "04b_joined_retention_task.json"   # produced by the extension described above
try:
    joined = ai.load_json(JOINED)["rows"]
except FileNotFoundError:
    joined = None
    print(f"{JOINED} not found — RQ3 cannot be computed yet. This is expected.")

if joined:
    table8 = []
    for name, key in [("retention half-life", "half_life"),
                      ("target rank @8K", "rank_at_8k"),
                      ("target mass @8K", "p_mem_at_8k"),
                      ("readout entropy @8K", "entropy_at_8k")]:
        xs = [r.get(key) for r in joined]
        ys = [r.get("delta_f1") for r in joined]
        st = ai.spearman(xs, ys)
        table8.append({"predictor": name, "outcome": "per-example dF1", **st})
        print(json.dumps(table8[-1], indent=2, default=str))

    if any("boundary_js" in r for r in joined):
        st = ai.spearman([r["boundary_js"] for r in joined],
                         [r["delta_f1"] for r in joined])
        table8.append({"predictor": "boundary JS divergence",
                       "outcome": "per-example dF1", **st,
                       "prior_work_range": [-0.09, 0.00]})
        print("\nprior work reported rho in [-0.09, 0.00] for this row")
    ai.save_json(table8, "05_table8_rq3.json")

    # Table 9
    t9 = ai.variance_decomposition(joined, "delta_f1",
                                   ["id", "cell", "scale", "stratum"])
    ai.save_json(t9, "05_table9_variance.json")
    print("\nTable 9:", json.dumps(t9, indent=2))


In [ ]:
print("figures written to:", FIGDIR)
for f in sorted(os.listdir(FIGDIR)):
    print("  ", f)
print("\ntables written to:", RUN)
for f in sorted(os.listdir(RUN)):
    if f.endswith(".json"):
        print("  ", f)
